In [ ]:
import sys
import os
from pathlib import Path

# Ajouter le répertoire parent au chemin Python
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.O1_preprocessing.loader import load_dicom_volume, load_mask

# 1. Exploration de la structure réelle des données
data_raw = Path("../data/raw")
scan_dirs = sorted([d for d in data_raw.glob("CT_scans_*") if d.is_dir()])
groundtruth_dir = data_raw / "GroundTruth" / "GroundTruth"

# Vérification du contenu du dossier GroundTruth
if groundtruth_dir.exists():
    files = list(groundtruth_dir.glob("*.mat"))
    print(f"📂 Exemples de fichiers trouvés dans GroundTruth: {[f.name for f in files[:5]]}")
else:
    print(f"❌ Le chemin {groundtruth_dir} n'existe pas !")

print(f"🔍 Dossiers trouvés: {[d.name for d in scan_dirs]}")

# 2. Récupération de tous les patients avec leurs chemins
patients_info = {}

for scan_dir in scan_dirs:
    patient_dirs = sorted([p for p in scan_dir.glob("PAT*") if p.is_dir()])
    for patient_dir in patient_dirs:
        patient_id = patient_dir.name
        
        # On cherche un fichier .mat qui contient l'ID (ex: 001 ou PAT001)
        short_id = patient_id.replace("PAT", "")
        
         # Recherche insensible à la casse et récursive
        mask_file = [f for f in groundtruth_dir.rglob("*.mat") if short_id in f.name]
        
        has_mask = len(mask_file) > 0
        patients_info[patient_id] = {
            'volume_path': str(patient_dir),
            'mask_path': str(mask_file[0]) if has_mask else None,
            'has_volume': True,
            'has_mask': has_mask
        }

print(f"\n✅ Total patients trouvés: {len(patients_info)}")

# 3. Filtrer les patients qui ont VOLUME + MASQUE
patients_with_both = [pid for pid, info in patients_info.items() 
                      if info['has_volume'] and info['has_mask']]

print(f"✅ Patients avec volume ET masque: {len(patients_with_both)}")

# 4. Charger les données pour tous les patients
stats_list = []

print("\n📊 Chargement des données...")

for patient_id in patients_with_both:
    try:
        vol_path = patients_info[patient_id]['volume_path']
        mask_path = patients_info[patient_id]['mask_path']
        
        # Charger le volume
        vol, metadata = load_dicom_volume(vol_path)
        
        # Charger le masque
        mask, mask_meta = load_mask(mask_path)
        
        if vol is not None and mask is not None:
            stats_list.append({
                'Patient_ID': patient_id,
                'Nb_Coupes': vol.shape[0],
                'Résolution_XY': f"{metadata['spacing'][0]:.2f}×{metadata['spacing'][1]:.2f}",
                'Epaisseur_Z': f"{metadata['slice_spacing']:.2f}",
                'Min_HU': f"{vol.min():.0f}",
                'Max_HU': f"{vol.max():.0f}",
                'Voxels_embolie': int(mask.sum()),
                'Pourcentage': f"{100*mask.sum()/mask.size:.2f}%"
            })
    except Exception as e:
        print(f"⚠️ Erreur chargement {patient_id}: {e}")

# 5. Affichage des résultats
if stats_list:
    df_stats = pd.DataFrame(stats_list)
    print("\n📊 RÉSUMÉ DU DATASET")
    display(df_stats)
    
    # Statistiques globales
    print(f"\n📈 Patients avec données complètes: {len(stats_list)}")
else:
    print("\n❌ Aucun patient avec data complète trouvé")